# L64 Proof Replay

Replays the bounded persona certification proof on fixed repo state.

Proof focuses on:
- Four ordered certification phases
- Five governance rules
- All 39 personas pass field completeness, coverage consistency, approval alignment
- Coverage partition: registry-backed + contract-only = total

In [ ]:
from pathlib import Path
import yaml

repo_root = Path.cwd()
contract = yaml.safe_load(
    (repo_root / 'registry' / 'persona_certification_v1.yaml').read_text(encoding='utf-8')
)
registry = yaml.safe_load(
    (repo_root / 'registry' / 'persona_registry_v2.yaml').read_text(encoding='utf-8')
)

assert len(contract['certification_phases']) == 4
assert set(contract['governance_rules'].keys()) == {
    'fail_closed', 'audit_completeness', 'bounded_claims',
    'determinism', 'coverage_partition',
}

personas = registry['personas']
REQUIRED_FIELDS = {
    'persona_id', 'display_name', 'slug', 'title', 'department', 'status',
    'coverage_status', 'risk_tier', 'approval_profile', 'approval_owner',
    'escalation_owner', 'manager', 'canonical_agent', 'external_presence_policy',
    'responsibilities', 'capability_families', 'workload_families',
    'allowed_domains', 'allowed_actions', 'action_count', 'aliases',
}

RISK_BASELINE = {
    'critical': 'critical-regulated',
    'high': 'high-impact',
    'medium': 'medium-operational',
    'low': 'low-observe-create',
}
STRICTNESS = {
    'low-observe-create': 0,
    'medium-operational': 1,
    'high-impact': 2,
    'critical-regulated': 3,
}

rb = 0
co = 0
for pid, p in personas.items():
    assert REQUIRED_FIELDS.issubset(set(p.keys()))
    baseline = RISK_BASELINE[p['risk_tier']]
    assert STRICTNESS[p['approval_profile']] >= STRICTNESS[baseline]
    if p['coverage_status'] == 'registry-backed':
        assert p['action_count'] > 0
        rb += 1
    else:
        assert p['action_count'] == 0
        co += 1

kpis = contract['kpis']
assert rb == kpis['registry_backed_personas']
assert co == kpis['contract_only_personas']
assert rb + co == kpis['total_personas']